# RECAST Pipeline
End-to-end: raw data → cleaning → augmentation → LoRA fine-tuning → cross-validation.

## Config

In [ ]:
from pathlib import Path

RAW_DATA       = Path("datasets/RECAST-30K.json")
CLEAN_DATA     = Path("datasets/recast_30k_clean.jsonl")
AUGMENTED_DATA = Path("datasets/recast_30k_augmented.jsonl")

## Step 1 — Cleaning

In [ ]:
from src.crllm.dataset.preprocess.preprocess import run_pipeline

clean_stats = run_pipeline(
    input_path=RAW_DATA,
    output_path=CLEAN_DATA,
    min_length=15,
    dedup_threshold=0.85,
    imbalance_threshold=0.5,
)
print(clean_stats)

## Step 2 — Augmentation

In [ ]:
from src.crllm.dataset.augmentation.augment import run_augmentation

aug_stats = run_augmentation(
    input_path=CLEAN_DATA,
    output_path=AUGMENTED_DATA,
    seed=42,
)
print(aug_stats)

## Step 3 — LoRA Fine-Tuning

**TODO:** Refactor `lora_finetuning_basic/lora_base.py` into a callable API, then wire it up here.

Needed: `run_lora_finetune(dataset_path, output_path, **config)` that accepts at minimum:
- `dataset_path` — path to the augmented JSONL
- `output_path` — where to save the adapter weights
- `model_name`, `lora_rank`, `lora_alpha`, `learning_rate`, `num_epochs`

```python
# from lora_finetuning_basic.lora_base import run_lora_finetune
#
# lora_stats = run_lora_finetune(
#     dataset_path=AUGMENTED_DATA,
#     output_path=Path("outputs/lora_adapter"),
#     model_name="meta-llama/Llama-3.2-1B-Instruct",
#     lora_rank=8,
#     learning_rate=1e-4,
#     num_epochs=1,
# )
# print(lora_stats)
```

## Step 4 — Cross-Validation

**TODO:** Refactor `Cross_validation/cross_validation_kfold.ipynb` into a callable API, then wire it up here.

Needed: `run_cross_validation(dataset_path, lora_adapter_path, **config)` that accepts at minimum:
- `dataset_path` — path to the evaluation dataset
- `lora_adapter_path` — path to the trained LoRA adapter from Step 3
- `k_folds`, `samples_per_fold`

Expected return structure — list of per-example dicts, one per model per fold:
```python
[
  {
    "fold": 1,
    "model": "LoRA",        # or "Full_FT"
    "csr": 0.72,            # fraction of constraint categories passed
    "Length": True,
    "Keyword": False,
    "Start_With": True,
    "End_With": True,
    "Format": True,
    "Tone": True,
    "Style": True,
    "Role_Playing": True,
    "Numerical_Constraints": True,
  },
  ...
]
```

```python
# from Cross_validation.cross_validation_kfold import run_cross_validation
#
# cv_results = run_cross_validation(
#     dataset_path=AUGMENTED_DATA,
#     lora_adapter_path=Path("outputs/lora_adapter"),
#     k_folds=5,
#     samples_per_fold=200,
# )
```

## Step 5 — Visualization

Compares LoRA vs Full Fine-Tuning across:
- Overall K-fold CSR (headline number)
- Per-constraint-category satisfaction rates
- Per-fold CSR (consistency check)

**Until Step 4 is wired up**, the cell below loads `cv_results` from a saved JSON if it exists,
otherwise falls back to mock data so the plots can be developed and inspected independently.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Load results ─────────────────────────────────────────────────────────────
# Replace this block with:  cv_results = run_cross_validation(...)
# once Step 4 is implemented.

RESULTS_JSON = Path("outputs/kfold_results/all_results.json")

if RESULTS_JSON.exists():
    with open(RESULTS_JSON) as f:
        cv_results = json.load(f)
    print(f"Loaded {len(cv_results)} results from {RESULTS_JSON}")
else:
    print("No results file found — using mock data for plot skeleton.")
    rng = np.random.default_rng(42)
    CATS = ["Length", "Keyword", "Start_With", "End_With",
            "Format", "Tone", "Style", "Role_Playing", "Numerical_Constraints"]
    # LoRA is modestly better on structured constraints; Full FT on Style/Tone
    lora_cat_p   = [0.78, 0.65, 0.82, 0.80, 0.74, 0.70, 0.72, 0.90, 0.88]
    full_ft_cat_p = [0.74, 0.63, 0.79, 0.76, 0.71, 0.75, 0.76, 0.90, 0.88]
    cv_results = []
    for fold in range(1, 6):
        for model, probs in [("LoRA", lora_cat_p), ("Full_FT", full_ft_cat_p)]:
            for _ in range(200):
                row = {"fold": fold, "model": model}
                passed = [bool(rng.random() < p) for p in probs]
                for cat, p in zip(CATS, passed):
                    row[cat] = p
                row["csr"] = sum(passed) / len(passed)
                cv_results.append(row)

In [ ]:
# ── Build summary dataframe ───────────────────────────────────────────────────
CATS = ["Length", "Keyword", "Start_With", "End_With",
        "Format", "Tone", "Style", "Role_Playing", "Numerical_Constraints"]
K_FOLDS = 5
MODELS  = ["LoRA", "Full_FT"]
COLORS  = {"LoRA": "#2196F3", "Full_FT": "#FF5722"}
LABELS  = {"LoRA": "LoRA", "Full_FT": "Full Fine-Tune"}

df = pd.DataFrame(cv_results)

def fold_means(model):
    """Return per-fold mean CSR and per-category rates for one model."""
    sub = df[df["model"] == model]
    rows = []
    for fold in range(1, K_FOLDS + 1):
        fdf = sub[sub["fold"] == fold]
        row = {"fold": fold, "csr": fdf["csr"].mean()}
        for cat in CATS:
            if cat in fdf.columns:
                row[cat] = fdf[cat].mean()
        rows.append(row)
    return pd.DataFrame(rows)

summaries = {m: fold_means(m) for m in MODELS}

# Overall K-fold mean ± std per model
overall = {
    m: {
        "mean": summaries[m]["csr"].mean(),
        "std":  summaries[m]["csr"].std(),
    }
    for m in MODELS
}

# Per-category mean ± std per model
cat_stats = {
    m: {
        cat: {
            "mean": summaries[m][cat].mean() if cat in summaries[m].columns else 0,
            "std":  summaries[m][cat].std()  if cat in summaries[m].columns else 0,
        }
        for cat in CATS
    }
    for m in MODELS
}

print("Overall CSR (mean ± std across 5 folds):")
for m in MODELS:
    print(f"  {LABELS[m]:>16}: {overall[m]['mean']:.4f} ± {overall[m]['std']:.4f}")

In [ ]:
# ── Plot 1: Per-fold CSR — consistency check ──────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fold_nums = list(range(1, K_FOLDS + 1))

for m in MODELS:
    vals = summaries[m]["csr"].values * 100
    ax.plot(fold_nums, vals, marker="o", label=LABELS[m],
            color=COLORS[m], linewidth=2, markersize=7)
    ax.fill_between(
        fold_nums,
        vals - summaries[m]["csr"].std() * 100,
        vals + summaries[m]["csr"].std() * 100,
        alpha=0.12, color=COLORS[m],
    )

ax.set_xlabel("Fold")
ax.set_ylabel("CSR (%)")
ax.set_title("Per-Fold CSR — LoRA vs Full Fine-Tune\n(stability across data splits)")
ax.set_xticks(fold_nums)
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/plot_fold_csr.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 2: Per-category satisfaction rates ───────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(CATS))
w = 0.35

for i, m in enumerate(MODELS):
    means  = [cat_stats[m][c]["mean"] * 100 for c in CATS]
    stds   = [cat_stats[m][c]["std"]  * 100 for c in CATS]
    offset = (i - 0.5) * w
    bars   = ax.bar(x + offset, means, w, yerr=stds, capsize=4,
                    label=LABELS[m], color=COLORS[m], alpha=0.85)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
                f"{mean:.0f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([c.replace("_", "\n") for c in CATS], fontsize=8)
ax.set_ylabel("Satisfaction Rate (%)")
ax.set_title("Per-Category CSR — LoRA vs Full Fine-Tune\n(mean ± std across 5 folds)")
ax.set_ylim(0, 120)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/plot_category_csr.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 3: Overall CSR — headline result ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))

means  = [overall[m]["mean"] * 100 for m in MODELS]
stds   = [overall[m]["std"]  * 100 for m in MODELS]
colors = [COLORS[m] for m in MODELS]
labels = [LABELS[m] for m in MODELS]

bars = ax.bar(labels, means, yerr=stds, capsize=8,
              color=colors, alpha=0.85, width=0.5)
for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + std + 1.5,
            f"{mean:.1f}%", ha="center", va="bottom",
            fontsize=13, fontweight="bold")

winner = LABELS[MODELS[int(means[1] > means[0])]]
ax.set_ylabel("CSR (%)")
ax.set_title(f"Overall K-Fold CSR\nWinner: {winner}")
ax.set_ylim(0, 120)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/plot_overall_csr.png", dpi=150, bbox_inches="tight")
plt.show()

delta = abs(means[0] - means[1])
print(f"\n{LABELS[MODELS[0]]}: {means[0]:.2f}% ± {stds[0]:.2f}%")
print(f"{LABELS[MODELS[1]]}: {means[1]:.2f}% ± {stds[1]:.2f}%")
print(f"Δ = {delta:.2f}pp  →  Winner: {winner}")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows = [{"Metric": "Overall CSR",
         "LoRA": f"{overall['LoRA']['mean']*100:.1f} ± {overall['LoRA']['std']*100:.1f}",
         "Full Fine-Tune": f"{overall['Full_FT']['mean']*100:.1f} ± {overall['Full_FT']['std']*100:.1f}",
         "Winner": LABELS[MODELS[int(overall['Full_FT']['mean'] > overall['LoRA']['mean'])]]}]

for cat in CATS:
    lm = cat_stats["LoRA"][cat]["mean"] * 100
    ls = cat_stats["LoRA"][cat]["std"]  * 100
    fm = cat_stats["Full_FT"][cat]["mean"] * 100
    fs = cat_stats["Full_FT"][cat]["std"]  * 100
    winner = "LoRA" if lm > fm else ("Full Fine-Tune" if fm > lm else "Tie")
    rows.append({"Metric": cat,
                 "LoRA": f"{lm:.1f} ± {ls:.1f}",
                 "Full Fine-Tune": f"{fm:.1f} ± {fs:.1f}",
                 "Winner": winner})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))